### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\ankit\AppData\Local\Temp\ipykernel_53876\3494988372.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
e:\AgenticAi\AgenticRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: 2406049_ANKIT_KUMAR_CSE.pdf
  ✓ Loaded 1 pages

Processing: Resume _ML_Offv1.pdf
  ✓ Loaded 1 pages

Processing: Resume_OFFv1.pdf
  ✓ Loaded 1 pages

Processing: Resume_OFFv2.pdf
  ✓ Loaded 1 pages

Total documents loaded: 4


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'iLovePDF', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-03-10T16:50:42+00:00', 'author': '', 'keywords': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'moddate': '2026-03-16T20:54:44+00:00', 'source': '..\\data\\pdf\\2406049_ANKIT_KUMAR_CSE.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '2406049_ANKIT_KUMAR_CSE.pdf', 'file_type': 'pdf'}, page_content='Ankit Kumar+91-7367871361\nBachelor of Technology ankitk.ug24.cs@nitp.ac.in\nComputer Science & Engineering linkedin.com/in/ankittkr21\nNational Institute of Technology, Patna github.com/Ankittkr\nEduca tion\nDegree/Certificate Institute/Board CGPA/Percentage Year\nB.Tech., CSE National Institute of Technology, Patna 8.56 2024-Present\nSenior Secondary British English School, Gaya (CBSE) 94.6 2022-2024\nSecondary British English School, Gaya (CBSE) 95.8 2020-2

In [8]:
### Text splitting get into chunks

def splitDocuments(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [15]:
chunks=splitDocuments(all_pdf_documents)
chunks

Split 4 documents into 13 chunks

Example chunk:
Content: Ankit Kumar+91-7367871361
Bachelor of Technology ankitk.ug24.cs@nitp.ac.in
Computer Science & Engineering linkedin.com/in/ankittkr21
National Institute of Technology, Patna github.com/Ankittkr
Educa t...
Metadata: {'producer': 'iLovePDF', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-03-10T16:50:42+00:00', 'author': '', 'keywords': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'moddate': '2026-03-16T20:54:44+00:00', 'source': '..\\data\\pdf\\2406049_ANKIT_KUMAR_CSE.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '2406049_ANKIT_KUMAR_CSE.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'iLovePDF', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-03-10T16:50:42+00:00', 'author': '', 'keywords': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'moddate': '2026-03-16T20:54:44+00:00', 'source': '..\\data\\pdf\\2406049_ANKIT_KUMAR_CSE.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '2406049_ANKIT_KUMAR_CSE.pdf', 'file_type': 'pdf'}, page_content='Ankit Kumar+91-7367871361\nBachelor of Technology ankitk.ug24.cs@nitp.ac.in\nComputer Science & Engineering linkedin.com/in/ankittkr21\nNational Institute of Technology, Patna github.com/Ankittkr\nEduca tion\nDegree/Certificate Institute/Board CGPA/Percentage Year\nB.Tech., CSE National Institute of Technology, Patna 8.56 2024-Present\nSenior Secondary British English School, Gaya (CBSE) 94.6 2022-2024\nSecondary British English School, Gaya (CBSE) 95.8 2020-2

### Embedding And vectorStoreDB

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from sklearn.metrics.pairwise import cosine_similarity
from typing import List , Dict , Tuple
import numpy as np

In [ ]:
class EmbeddingManager :
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self , model_name='all-MiniLM-L6-v2') :
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name=model_name
        self.model = None
        self._load_model()

    def _load_model(self) : 
        """Load the SentenceTransformer model"""

        try :
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    
    def generate_embeddings(self , texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        embeddings  = self.model.encode(texts , show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings 
    

## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager
    

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3203.92it/s]


Model loaded successfully. Embedding dimension: 384


In [14]:
class VectorStore:
    def __init__(self , collection_name:str='pdf_document' , persist_directory:str='../data/vector_store'):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory , exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            self.collection= self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self , documents:List[any] , embeddings:np.ndarray ):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
vectorstore=VectorStore()
vectorstore




Vector store initialized. Collection: pdf_document
Existing documents in collection: 0


In [16]:
chunks

[Document(metadata={'producer': 'iLovePDF', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-03-10T16:50:42+00:00', 'author': '', 'keywords': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'moddate': '2026-03-16T20:54:44+00:00', 'source': '..\\data\\pdf\\2406049_ANKIT_KUMAR_CSE.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '2406049_ANKIT_KUMAR_CSE.pdf', 'file_type': 'pdf'}, page_content='Ankit Kumar+91-7367871361\nBachelor of Technology ankitk.ug24.cs@nitp.ac.in\nComputer Science & Engineering linkedin.com/in/ankittkr21\nNational Institute of Technology, Patna github.com/Ankittkr\nEduca tion\nDegree/Certificate Institute/Board CGPA/Percentage Year\nB.Tech., CSE National Institute of Technology, Patna 8.56 2024-Present\nSenior Secondary British English School, Gaya (CBSE) 94.6 2022-2024\nSecondary British English School, Gaya (CBSE) 95.8 2020-2

In [17]:
### Convert the text to embeddings
texts=[ doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.75s/it]


Generated embeddings with shape: (13, 384)
Adding 13 documents to vector store...
Successfully added 13 documents to vector store
Total documents in collection: 13


### Retriever Pipeline From VectorStore

In [18]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [19]:
rag_retriever

In [35]:
rag_retriever.retrieve(""" who is an aspiring Computer Science student to innovative software projects. Motivated to
learn, collaborate,  grow fastpaced development environments""" )

Retrieving documents for query: ' who is an aspiring Computer Science student to innovative software projects. Motivated to
learn, collaborate,  grow fastpaced development environments'
Top K: 5, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.26it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_742dcb1d_3',
  'content': 'Ankit Kumar\nSoftware Developer\n♂phone+91-7367871361/envel⌢pekr.ankitt21@gmail.com/linkedinlinkedin.com/in/ankitkumar\nSummary\nI am a Computer Science student with a strong foundation in programming, data structures along with a growing\ninterest in Machine Learning and Artificial Intelligence. I am seeking a challenging role to my technical skills to\ndevelop intelligent systems and AI-powered applications. I am highly motivated to learn advanced ML/AI concepts,\ncollaborate with teams, and grow in fast-paced, innovation-driven environments.”\nSkills\nProgramming Languages:Python, C++\nMachine Learning:Supervised Learning, Unsupervised Learning, Model Evaluation, Feature Engineering,\nRegression, Classification\nLibraries / T ools:NumPy, Pandas, Matplotlib, Scikit-learn\nData Handling:Data Cleaning, Data Preprocessing, Exploratory Data Analysis (EDA)\nMathematics:Linear Algebra, Probability, Statistics\nT ools & Platforms:Jupyter Notebook, Goo

### RAG Pipeline- VectorDB To LLM Output Generation

In [37]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

from langchain_groq import ChatGroq


In [ ]:
llm=ChatGroq(
    model='openai/gpt-oss-20b',
    temperature=0.1
    )

def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt = f"""
    You are an AI assistant that answers questions about a candidate based only on the provided resume context.

    Instructions:
    - Answer using only the information present in the context.
    - Be concise, professional, and accurate.
    - If the answer is not available in the context, say:
    "I could not find that information in the resume."
    - Do not make assumptions or invent details.
    - For skills, projects, education, or experience questions, summarize the relevant information clearly.
    - Use bullet points when appropriate.

    Resume Context:
    {context}

    Question:
    {query}

    Answer:
    """
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [41]:
query=""" who is an aspiring Computer Science student to innovative software projects. Motivated to
learn, collaborate,  grow fastpaced development environments. Tell his skills and about his projects"""
answer=rag_simple(query , rag_retriever , llm  )
answer

Retrieving documents for query: ' who is an aspiring Computer Science student to innovative software projects. Motivated to
learn, collaborate,  grow fastpaced development environments. Tell his skills and about his projects'
Top K: 3, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.77it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


'**Ankit\u202fKumar** – an aspiring Computer Science student focused on innovative software projects.  \n\n**Key skills**  \n- **Programming languages:** C, C++, Java, JavaScript, Python  \n- **Web technologies:** HTML, CSS, React.js, Node.js, Express.js, MongoDB, Tailwind\u202fCSS  \n- **Tools & platforms:** Git, Postman, VS\u202fCode  \n\n**Projects & experience**  \n- **Frontend Lead – Web Development Cell, NIT Patna (Mar\u202f2026\u202f–\u202fpresent)** – led the frontend team on multiple student‑driven web projects, delivering responsive, user‑friendly UIs and integrating REST APIs with backend services.  \n- (Earlier résumé highlights ML/AI interests and Python‑based data‑science skills, but the current focus is on full‑stack web development.)'

### Enhanced RAG Pipeline Features

In [44]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("""" who is an aspiring Computer Science student to innovative software projects. Motivated to learn, collaborate,  grow fastpaced development environments.""", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: '" who is an aspiring Computer Science student to innovative software projects. Motivated to learn, collaborate,  grow fastpaced development environments.'
Top K: 3, Score threshold: 0.1


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Answer: Ankit Kumar.
Sources: [{'source': 'Resume _ML_Offv1.pdf', 'page': 0, 'score': 0.13384199142456055, 'preview': 'Ankit Kumar\nSoftware Developer\n♂phone+91-7367871361/envel⌢pekr.ankitt21@gmail.com/linkedinlinkedin.com/in/ankitkumar\nSummary\nI am a Computer Science student with a strong foundation in programming, data structures along with a growing\ninterest in Machine Learning and Artificial Intelligence. I am se...'}]
Confidence: 0.13384199142456055
Context Preview: Ankit Kumar
Software Developer
♂phone+91-7367871361/envel⌢pekr.ankitt21@gmail.com/linkedinlinkedin.com/in/ankitkumar
Summary
I am a Computer Science student with a strong foundation in programming, data structures along with a growing
interest in Machine Learning and Artificial Intelligence. I am se


In [46]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("who is an aspiring Computer Science student to innovative software projects. Motivated to learn, collaborate,  grow fastpaced development environments", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'who is an aspiring Computer Science student to innovative software projects. Motivated to learn, collaborate,  grow fastpaced development environments'
Top K: 3, Score threshold: 0.1


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.63it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Ankit Kumar
Software Developer
♂phone+91-7367871361/envel⌢pekr.ankitt21@gmail.com/linkedinl

inkedin.com/in/ankitkumar
Summary
I am a Computer Science student with a strong foundation in programming, data structures along with a growing
interest in Machine Learning and Artificial Intelligence. I am seeking a challenging role to my technical skills to
develop intelligent systems and AI-powered applications. I am highly motivated to learn advanced ML/AI concepts,
collaborate with teams, and grow in fast-paced, innovation-driven environments.”
Skills
Programming Languages:Python, C++
Machine Learning:Supervised Learning, Unsupervised Learning, Model Evaluation, Feature Engineering,
Regression, Classification
Libraries / T ools:NumPy, Pandas, Matplotlib, Scikit-learn
Data Handling:Data Cleaning, Data Preprocessing, Exploratory Data Analysis (EDA)
Mathematics:Linear Algebra, Probability, Statistics
T ools & Platforms:Jupyter Notebook, Google Colab, Git & GitHub
Coding Profiles

Question: who is an aspiring Computer Science student to innovative software projects. Motivated to learn